In [1]:
using Pkg
Pkg.instantiate()
Pkg.activate("..")
Pkg.develop(path="C:/Users/ahamed/.julia/dev/myExample")
using Random
using LinearAlgebra
using Printf
using Plots
using myExample
using JLD2, Statistics
const MF = myExample.MiniFlux
const AD =  myExample.AutoDiff

Precompiling project...
    646.6 ms  ? myExample
  Activating project at `c:\Users\ahamed\.julia\dev\myExample\examples\CNN`
   Resolving package versions...
  No Changes to `C:\Users\ahamed\.julia\dev\myExample\examples\CNN\Project.toml`
  No Changes to `C:\Users\ahamed\.julia\dev\myExample\examples\CNN\Manifest.toml`


myExample.AutoDiff

In [2]:
println("Loading dataset...")
X_train = load("../data/imdb_dataset_prepared.jld2", "X_train")
y_train = load("../data/imdb_dataset_prepared.jld2", "y_train")
X_test = load("../data/imdb_dataset_prepared.jld2", "X_test")
y_test = load("../data/imdb_dataset_prepared.jld2", "y_test")
embeddings = load("../data/imdb_dataset_prepared.jld2", "embeddings")
vocab = load("../data/imdb_dataset_prepared.jld2", "vocab")

Loading dataset...


12849-element Vector{String}:
 "confined"
 "dumber"
 "henry"
 "abducted"
 "rises"
 "progression"
 "il"
 "gathered"
 "lovers"
 "cannibalistic"
 ⋮
 "poetic"
 "ponderous"
 "maybe"
 "towel"
 "uncut"
 "joint"
 "treacherous"
 "dev"
 "<pad>"

In [3]:
train_data = [(X_train[i], y_train[i]) for i in 1:length(y_train)]
test_data = [(X_test[i], y_test[i]) for i in 1:length(y_test)]
println("Dataset loaded with $(length(train_data)) training samples")

Dataset loaded with 40000 training samples


In [4]:
# Training loop
function print_data_shapes()
    println("Input data shapes:")
    println("X_train shape: ", size(X_train))
    println("y_train shape: ", size(y_train))
    println("X_test shape: ", size(X_test))
    println("y_test shape: ", size(y_test))
    println("embeddings shape: ", size(embeddings))
    println("vocab shape: ", size(vocab))
    println("train_data shape: ", size(train_data))
    println("test_data shape: ", size(test_data))
end

print_data_shapes (generic function with 1 method)

In [5]:
print_data_shapes()

Input data shapes:
X_train shape: (130, 40000)
y_train shape: (1, 40000)
X_test shape: (130, 10000)
y_test shape: (1, 10000)
embeddings shape: (50, 12849)
vocab shape: (12849,)
train_data shape: (40000,)
test_data shape: (10000,)


In [ ]:
embedding_dim = size(embeddings, 1)
vocab_size    = length(vocab)

model = MF.Model([
    MF.Embedding(vocab_size, embedding_dim),
    MF.PermuteDims((2,1,3)),
    MF.Conv1D(3, embedding_dim, 8, activation=MF.relu),
    MF.MaxPool1D(8),
    MF.Flatten(),
    MF.Dense(128, 1, AD.σ)    # Twoja Dense z AutoDiff
])

myExample.MiniFlux.Model(myExample.MiniFlux.Embedding[myExample.MiniFlux.Embedding(var W_embed
 ┣━ ^ 50×12849 Matrix{Float64}
 ┗━ ∇ Nothing)], myExample.AutoDiff.Variable[var W_embed
 ┣━ ^ 50×12849 Matrix{Float64}
 ┗━ ∇ Nothing])

In [7]:
model.layers[1].weight.output .= embeddings'

using Optimisers

# Accuracy Function
function accuracy_fn(y::AD.GraphNode, ŷ::AD.GraphNode)
    y_float = AD.BroadcastedOperator(Float32, y)
    preds = AD.BroadcastedOperator(>, ŷ, AD.Constant(0.5f0))
    truths = AD.BroadcastedOperator(>, y_float, AD.Constant(0.5f0))
    matches = AD.BroadcastedOperator(==, preds, truths)
    return AD.mean(AD.BroadcastedOperator(Float32, matches))
end

opt = Adam()



ErrorException: type Embedding has no field weight

In [8]:

# Create batches
batchsize = 64
train_loader = MF.create_batches(X_train, y_train, batchsize)
test_loader = MF.create_batches(X_test, y_test, batchsize)

# Training loop
epochs = 5
for epoch in 1:epochs
    ## --- training ---
    train_loss = 0.0
    train_acc  = 0.0
    n_batches  = 0

    for (xs,ys) in train_loader
        # Process batch

        println("xs: ", size(xs))
        println("ys: ", size(ys))
        # Convert to AD variables
        x_var = AD.Variable(xs)
        y_var = AD.Variable(ys)

        # Forward pass
        ŷ_var = model(x_var)
        println("ŷ_var: ", ŷ_var)
        l_var = MF.binary_cross_entropy_loss(y_var, ŷ_var)
        println("l_var: ", l_var)
        g = AD.topological_sort(l_var)
        AD.forward!(g)
        AD.backward!(g)

        # Collect gradients
        grads = map(p->p.gradient, model.params)

        # Update parameters
        opt_state = Optimisers.setup(opt, model.params)
        Optimisers.update!(opt_state, model.params, grads)

        # Zero gradients
        for p in model.params
            p.gradient .= zero(p.gradient)
        end

        # Update metrics
        train_loss += l_var
        train_acc  += accuracy_fn(y_var.output, ŷ_var.output)
        n_batches  += 1
    end

    # Average metrics
    train_loss /= n_batches
    train_acc  /= n_batches

    ## --- evaluation ---
    test_loss = 0.0
    test_acc  = 0.0
    m_batches = 0

    for batch in test_loader
        # Process batch
        xs = hcat(first.(batch)...)
        ys = hcat(last.(batch)...)

        # Forward pass only
        ŷ = model(AD.Constant(xs)).output
        l  = MF.binary_cross_entropy_loss(ys, ŷ)

        # Update metrics
        test_loss += l
        test_acc  += accuracy_fn(ys, ŷ)
        m_batches += 1
    end

    # Average metrics
    test_loss /= m_batches
    test_acc  /= m_batches

    # Print progress
    @printf("Epoch %d \tTrain: loss=%.4f, acc=%.4f \tTest: loss=%.4f, acc=%.4f\n",
            epoch, train_loss, train_acc, test_loss, test_acc)
end

xs: (130, 64)
ys: (1, 64)
layer: myExample.MiniFlux.Embedding(var W_embed
 ┣━ ^ 50×12849 Matrix{Float64}
 ┗━ ∇ Nothing)
a: myExample.AutoDiff.EmbeddingOperator
ŷ_var: op.embedding(Embedding
 ┣━ W: W_embed
 ┣━ x: 130×64 Matrix{Int64}
 ┣━ output: Nothing
 ┗━ gradient: Nothing
l_var: op.?(typeof(*))


MethodError: MethodError: no method matching forward(::myExample.AutoDiff.EmbeddingOperator, ::Matrix{Float64}, ::Matrix{Int64})
The function `forward` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  forward(!Matched::myExample.AutoDiff.BroadcastedOperator{typeof(-)}, ::Any, ::Any)
   @ myExample C:\Users\ahamed\.julia\dev\myExample\src\AutoDiff\operators.jl:25
  forward(!Matched::myExample.AutoDiff.BroadcastedOperator{typeof(+)}, ::Any, ::Any)
   @ myExample C:\Users\ahamed\.julia\dev\myExample\src\AutoDiff\operators.jl:29
  forward(!Matched::myExample.AutoDiff.BroadcastedOperator{typeof(/)}, ::Any, ::Any)
   @ myExample C:\Users\ahamed\.julia\dev\myExample\src\AutoDiff\operators.jl:33
  ...
